In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Basés sur les ratios mesurés : BG/TC=203x, BG/AR=17x
# α/β ≈ 203/17 ≈ 12
ALPHA = 12.0
BETA  = 1.0
 
KEEP_PERCENTAGES = [0.25, 0.33, 0.50, 0.75]

In [4]:
# chargement du CSV
try:
    df = pd.read_csv('../distribution_csv_files/train_class_distribution.csv')
    print(f'CSV chargé: {len(df)} fichiers')
except FileNotFoundError:
    raise FileNotFoundError("Fichier introuvable")

CSV chargé: 398 fichiers


In [5]:
# Calcul du score pondéré
df['score'] = ALPHA * df['pct_tc'] + BETA * df['pct_ar']
df = df.sort_values('score', ascending=False).reset_index(drop=True)
 

print('Statistique')
print(f'Min: {df["score"].min():.4f}')
print(f'Max: {df["score"].max():.4f}')
print(f'Moyenne: {df["score"].mean():.4f}')
print(f'Médiane: {df["score"].median():.4f}')
print(f'Std: {df["score"].std():.4f}')
print()

Statistique
Min: 1.4933
Max: 37.0542
Moyenne: 11.2159
Médiane: 10.3889
Std: 5.4988



In [8]:
results = []
for pct in KEEP_PERCENTAGES:
    n_keep  = int(len(df) * pct)
    subset  = df.head(n_keep)
    total_tc = subset['n_tc'].sum()
    total_ar = subset['n_ar'].sum()
    total_bg = subset['n_bg'].sum()
    results.append({
        'pct_kept': pct,
        'n_keep': n_keep,
        'threshold': subset['score'].min(),
        'pct_tc_kept': 100 * total_tc / df['n_tc'].sum(),
        'pct_ar_kept': 100 * total_ar / df['n_ar'].sum(),
        'ratio_tc': total_bg / max(total_tc, 1),
        'ratio_ar': total_bg / max(total_ar, 1),
    })

print(f'{"% fichiers":<12} {"N":<8} {"% TC gardé":<14} {"% AR gardé":<14} {"BG/TC":<10} {"BG/AR"}')
print('-' * 67)
for r in results:
    print(f'{100*r["pct_kept"]:>6.0f}%      {r["n_keep"]:<8} '
          f'{r["pct_tc_kept"]:>8.1f}%      {r["pct_ar_kept"]:>8.1f}%      '
          f'{r["ratio_tc"]:>6.0f}x    {r["ratio_ar"]:.0f}x')

% fichiers   N        % TC gardé     % AR gardé     BG/TC      BG/AR
-------------------------------------------------------------------
    25%      99           53.2%          30.4%          93x    13x
    33%      131          63.3%          39.0%         104x    14x
    50%      199          80.9%          56.3%         124x    15x
    75%      298          95.4%          81.4%         158x    15x


In [ ]:
# On veux trouver un seuil pour lequel on peut garder 90% des pixels de TC puisqu'il s'agit de la classe la plus sous représentée
 
for pct in np.arange(0.10, 1.01, 0.01):
    n_keep = int(len(df) * pct)
    subset = df.head(n_keep)

    pct_tc_kept = 100 * subset['n_tc'].sum() / df['n_tc'].sum()
    pct_ar_kept = 100 * subset['n_ar'].sum() / df['n_ar'].sum()

    if pct_tc_kept >= 90.0:
        print(f'{100*pct:.0f}% des fichiers sont nécéssaire pour garder 90% des pixels TC')

        selected = df.head(n_keep)[['filename', 'score', 'pct_tc', 'pct_ar']]
        selected.to_csv(f'../distribution_csv_files/selected_files_top{100*pct:.0f}pct.csv', index=False)
        
        print(f'Liste sauvegardée dans selected_files_top{100*pct:.0f}pct.csv')
        break

64% des fichiers sont nécéssaire pour garder 90% des pixels TC
Liste sauvegardée dans selected_files_top64pct.csv


In [15]:
import pandas as pd

df = pd.read_csv('../distribution_csv_files/train_class_distribution.csv')

ALPHA, BETA = 12.0, 1.0
df['score'] = ALPHA * df['pct_tc'] + BETA * df['pct_ar']
df = df.sort_values('score', ascending=False).reset_index(drop=True)

n_keep = int(len(df) * 0.75)

selected = df.head(n_keep)[['filename', 'score', 'pct_tc', 'pct_ar']]
selected.to_csv('../distribution_csv_files/selected_files_top75pct.csv', index=False)
print(f'{n_keep} fichiers sauvegardés pour une selection de 75% des fichiers')


298 fichiers sauvegardés pour une selection de 75% des fichiers
